# Free 150-epoch training on Kaggle (background run)

Trains the full 150-epoch SwinUNETR on a **free Kaggle GPU** and keeps going with your browser closed.
Strategy, timings and the reasons behind each choice: `docs/reference/free-training-150-epochs.md`.

**One-time setup (right-hand *Notebook options* / *Session options* panel):**

1. **Accelerator:** `GPU T4 x2` (the code uses one T4). Kaggle has retired the P100, and current PyTorch builds no longer support it anyway; the GPU check below stops early on anything older than a T4.
2. **Internet: On.** Needed to clone the repo and download the dataset. Kaggle asks for phone verification the first time.
3. **Persistence:** leave as *No persistence*. Checkpoints are saved as this notebook's *Output* instead.

**To run it in the background:** click **Save Version → Save & Run All (Commit)**. It then runs on Kaggle's servers
even if you close the tab. Do **not** just press Run All in the editor: that session stops when you leave.

**Time limits:** a session lasts at most 12 hours. 150 epochs should take roughly 5 to 6 hours, so one session is usually enough.
If it is not, training stops cleanly at the budget below and you resume in a second session (see the *Resume* cell).

**Or push it from your machine** (no browser needed): `scripts/kaggle_push.sh` uploads this notebook with GPU T4 + Internet enabled and starts the commit run. See `docs/reference/free-training-150-epochs.md`.

## 1. Settings (edit before *Save & Run All*)

In [ ]:
BRANCH = 'train/150-epoch-free-run'   # branch that contains --resume support (use 'main' once it is merged)
EPOCHS = 150                          # must stay identical across every resumed session
EXTRA_FLAGS = '--use-checkpoint'      # same flags as the 20-epoch run; '' is ~30% faster but may run out of GPU memory
TRAIN_TIME_BUDGET_H = 11.0            # stop cleanly before Kaggle's 12 h session cap; resume in the next session

## 2. Check the GPU

In [ ]:
!nvidia-smi

## 3. Clone the repository and install dependencies
Kaggle already ships a CUDA build of PyTorch, NumPy, PyYAML, tqdm and matplotlib, so only MONAI, nibabel and einops are installed. TensorRT is **not** installed here: training and ONNX export do not need it.
The TensorRT benchmark is run afterwards on Colab with the `best.pt` this notebook produces.

In [ ]:
%cd /kaggle/working
!rm -rf swinunetr-spleen-tensorrt
!git clone --branch {BRANCH} https://github.com/KshitijV97/swinunetr-spleen-tensorrt.git
%cd /kaggle/working/swinunetr-spleen-tensorrt
# Only the training dependencies, with the same pins as requirements-colab.txt.
# The full Colab file also pins onnx/onnxruntime/pytest/requests, which would replace Kaggle's
# preinstalled copies (and can break other packages) without being needed for training.
!pip install -q monai==1.6.1 nibabel==5.4.2 einops==0.8.2
# The dataset (~1.5 GB) goes to Kaggle's scratch space so it is NOT saved as notebook output.
!mkdir -p /kaggle/temp/data && rm -rf data && ln -s /kaggle/temp/data data

In [ ]:
import os, subprocess, glob, shutil, json
from pathlib import Path
import torch, monai
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Set Accelerator = GPU T4 x2 in the session options.')
cap = torch.cuda.get_device_capability(0)
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0), '| sm', cap)
if cap < (7, 0):
    raise RuntimeError(f'GPU compute capability {cap} is too old for this PyTorch build. Use GPU T4 x2.')
print('monai', monai.__version__)

NOISE = ('nvtiff', 'nvjpeg2k', 'nvimagecodec')

def run(cmd, step, ok_codes=(0,)):
    """Run a shell step, stream and log its output, and stop the notebook if it fails."""
    log = Path('results/logs') / f'{step}.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f'$ {cmd}   (log: {log})', flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    with open(log, 'a') as f:
        proc = subprocess.Popen(cmd, shell=True, env=env, text=True, bufsize=1,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            f.write(line)
            if not any(n in line.lower() for n in NOISE):
                print(line, end='', flush=True)
    code = proc.wait()
    if code not in ok_codes:
        raise RuntimeError(f'Step "{step}" failed with exit code {code}. See {log}.')
    return code

## 4. Resume from a previous session (only for the 2nd, 3rd... session)
If an earlier run of this notebook stopped before epoch 150, open **Add Input → Notebook Output Files → *your earlier version***.
This cell finds its `last.pt` and copies it into `checkpoints/`. With no earlier output it does nothing and training starts at epoch 1.

In [ ]:
Path('checkpoints').mkdir(exist_ok=True)
found = sorted(glob.glob('/kaggle/input/**/last.pt', recursive=True), key=os.path.getmtime)
if found:
    shutil.copy(found[-1], 'checkpoints/last.pt')
    for extra in ('best.pt',):
        src = Path(found[-1]).with_name(extra)
        if src.exists():
            shutil.copy(src, f'checkpoints/{extra}')
    print('Found previous state:', found[-1])
else:
    print('No previous last.pt in /kaggle/input: starting from epoch 1.')

## 5. Download the dataset (~1-3 minutes)

In [ ]:
run('python -m src.data --download --summary', 'data')

## 6. Train (the long step)
Runs `python -m src.train --epochs 150 ...` under `timeout`, so it stops itself before Kaggle's session limit.
`last.pt` is written atomically after every epoch, so a stop loses at most one epoch.
Exit code 124 means "time budget reached, resume in the next session"; it is not an error.

In [ ]:
resume_flag = '--resume' if os.path.exists('checkpoints/last.pt') else ''
budget_s = int(TRAIN_TIME_BUDGET_H * 3600)
train_error = None
try:
    code_ = run(f'timeout {budget_s} python -m src.train --epochs {EPOCHS} {EXTRA_FLAGS} {resume_flag}',
                'train_resume' if resume_flag else 'train', ok_codes=(0, 124))
    if code_ == 124:
        print('\nTime budget reached before epoch', EPOCHS, '. Start a new version with this one added as an'
              ' input (Add Input -> Notebook Output Files) to continue.')
    else:
        print('\nTraining finished all', EPOCHS, 'epochs.')
except RuntimeError as e:
    # Keep going so the summary and cleanup cells still run; the last cell re-raises.
    train_error = e
    print('\nTRAINING FAILED:', e)


## 7. Summary and training curve

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
h = json.load(open('results/training_history.json')) if os.path.exists('results/training_history.json') else None
if h is None and os.path.exists('checkpoints/last.pt'):   # stopped early: history is inside last.pt
    h = torch.load('checkpoints/last.pt', map_location='cpu', weights_only=False)['history']
    print('(partial run: history read from last.pt)')
if h is None:
    print('No training history yet (training failed before the first epoch).')
else:
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(h['epoch'], h['train_loss'], color='tab:blue'); ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss')
    ax2 = ax1.twinx(); ax2.plot(h['val_epoch'], h['val_dice'], 'o-', color='tab:orange')
    ax2.set_ylabel('validation Dice'); ax2.set_ylim(0, 1)
    fig.tight_layout(); fig.savefig('results/training_curve.png', dpi=150)
    print('epochs done:', h['epoch'][-1], '| validation Dice so far:', [round(x, 3) for x in h['val_dice']])

## 7b. Clean up the output and report failures


In [ ]:
# Remove the dataset symlink so Kaggle does not try to save /kaggle/temp as output,
# then list what will be saved.
if os.path.islink('data'):
    os.unlink('data')
for f in sorted(glob.glob('checkpoints/*') + glob.glob('results/*.json') + glob.glob('results/*.png')):
    print(f'{os.path.getsize(f)/1e6:8.1f} MB  {f}')
if train_error is not None:
    raise train_error


## 8. What to do with the output
When the run finishes, open the saved version's **Output** tab and download:

- `swinunetr-spleen-tensorrt/checkpoints/best.pt` (the model you will benchmark)
- `swinunetr-spleen-tensorrt/results/training_history.json` and `training_curve.png`

Then follow *"After training"* in `docs/reference/free-training-150-epochs.md` to run export + the TensorRT benchmark on Colab (about 30 minutes).